# Определение занятости парковочных мест

Детекция свободных и занятых мест на полном кадре с помощью YOLO11n. Используются исходные кадры CNRPark+EXT и подготовленная из них выборка. Каждый эксперимент начинается с предобученных весов YOLO11n.

In [ ]:
from pathlib import Path
import pandas as pd
import torch, ultralytics
from ultralytics import YOLO
from IPython.display import display

PROJECT = Path('/Users/alikhansagat/Desktop/Ali/DeepLearning/Parking_Defense')
DATA = Path('/Users/alikhansagat/Desktop/Ali/DeepLearning/Parking_From_Zero/yolo_dataset/dataset.yaml')
RUNS = PROJECT / 'runs'
DEVICE = 0 if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Ultralytics:', ultralytics.__version__, 'PyTorch:', torch.__version__, 'Device:', DEVICE)
print(DATA.read_text())

## Данные

1600 кадров: 1120 для обучения, 240 для валидации, 240 для теста. Даты не пересекаются между выборками. Разметка содержит рамку и класс каждого места: 0 — свободно, 1 — занято. Камеры и парковка общие, поэтому оценка на новых парковках не проводилась.

## План экспериментов

Четыре запуска имеют одинаковый seed, 15 эпох, оптимизатор AdamW и исходные веса. E1 — базовый; в остальных изменяется один параметр относительно базового. Модель выбирается по validation mAP50-95. Тест используется один раз для выбранной модели.

In [ ]:
experiments = [
    {'name': 'E1_baseline', 'imgsz': 640, 'batch': 4, 'lr0': 0.001},
    {'name': 'E2_image_size', 'imgsz': 960, 'batch': 4, 'lr0': 0.001},
    {'name': 'E3_learning_rate', 'imgsz': 640, 'batch': 4, 'lr0': 0.0005},
    {'name': 'E4_batch_size', 'imgsz': 640, 'batch': 8, 'lr0': 0.001},
]
display(pd.DataFrame(experiments))

## Обучение

Эпоха — один проход по обучающим данным. Learning rate задаёт размер шага обновления весов, batch — число снимков за один проход модели. Каждый запуск сохраняет графики, метрики и лучшие веса в отдельной папке.

In [ ]:
run_dirs = []
for exp in experiments:
    name = exp['name']
    print(f"\n=== {name} ===")
    model = YOLO('yolo11n.pt')
    model.train(data=str(DATA), epochs=15, optimizer='AdamW', lr0=exp['lr0'],
                batch=exp['batch'], imgsz=exp['imgsz'], seed=1409,
                deterministic=True, device=DEVICE, workers=0, patience=15,
                project=str(RUNS), name=name, plots=True)
    run_dirs.append(Path(model.trainer.save_dir))

## Сравнение на валидационной выборке

Precision — доля верных среди найденных мест; recall — доля найденных среди размеченных. mAP50 и mAP50-95 учитывают класс и точность расположения рамки. Выбор лучшей модели основан на mAP50-95.

In [ ]:
summary = []
for exp, run in zip(experiments, run_dirs):
    model = YOLO(str(run / 'weights' / 'best.pt'))
    metrics = model.val(data=str(DATA), split='val', imgsz=exp['imgsz'], batch=exp['batch'], device=DEVICE, workers=0, plots=False)
    summary.append({**exp, 'precision': metrics.box.mp, 'recall': metrics.box.mr,
                    'mAP50': metrics.box.map50, 'mAP50-95': metrics.box.map,
                    'run_dir': str(run)})
results = pd.DataFrame(summary).sort_values('mAP50-95', ascending=False)
results.to_csv(PROJECT / 'comparison.csv', index=False)
display(results.round(4))

## Финальная оценка на тесте

Тестовая выборка не участвует в выборе настроек. Оцениваем на ней только победителя, выбранного по валидации.

In [ ]:
winner = results.iloc[0]
model = YOLO(str(Path(winner['run_dir']) / 'weights' / 'best.pt'))
test = model.val(data=str(DATA), split='test', imgsz=int(winner['imgsz']), batch=int(winner['batch']), device=DEVICE, workers=0, plots=True, project=str(RUNS), name='final_test')
test_results = pd.DataFrame([{'experiment': winner['name'], 'precision': test.box.mp, 'recall': test.box.mr, 'mAP50': test.box.map50, 'mAP50-95': test.box.map}])
test_results.to_csv(PROJECT / 'test_results.csv', index=False)
display(test_results.round(4))

## Пример предсказания и подсчёт мест

Рамки с классом free считаются свободными. Порог уверенности влияет на число обнаружений; результат зависит от качества разметки и условий съёмки.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
sample = next((DATA.parent / 'images' / 'test').glob('*.jpg'))
prediction = model.predict(source=str(sample), imgsz=int(winner['imgsz']), conf=0.25, device=DEVICE, verbose=False)[0]
classes = prediction.boxes.cls.cpu().numpy().astype(int)
print('Всего найдено:', len(classes), 'Свободно:', (classes == 0).sum(), 'Занято:', (classes == 1).sum())
plt.figure(figsize=(14, 9)); plt.imshow(prediction.plot()[:, :, ::-1]); plt.axis('off'); plt.show()